### C/ Spark : parallelisation of the image processing algorithm « MedianFilter »

In [1]:
import pyspark
from pyspark import SparkContext
import imageio
import os
import numpy as np
def readImg(path):
    img = imageio.imread(path)
    im = np.array(img,dtype='uint8')
    return im

def writeImg(path,buf):
    imageio.imwrite(path,buf)

def split_channel_images(channel_img, nx, nb_partitions):
    data=[]
    begin=0
    block_size=nx/nb_partitions
    for ip in range(nb_partitions):
        end=min(begin+block_size,nx)
        data.append([ip,begin,end,channel_img])
        begin=end
    return data

def compute_image_from_rdd(result_data, nb_partitions, nx, ny):
    new_img_buf=np.zeros((nx,ny),dtype='uint8')
    for part_id, part_buf in result_data:
        start = part_id * (nx // nb_partitions)
        end = start + part_buf.shape[0]
        new_img_buf[start:end, :] = part_buf
    return new_img_buf

def part_median_filter(local_data):
    part_id = int(local_data[0])
    first   = int(local_data[1])
    end     = int(local_data[2])
    buf     = local_data[3]
    nx=buf.shape[0]
    ny=buf.shape[1]
    
    ########################################
    #
    # CREATE NEW BUF WITH MEDIAN FILTER SOLUTION
    #
    new_buf=np.zeros((end-first,ny),dtype='uint8')
    print("new_buf = ",new_buf.shape)
    
    ##########################################
    #
    # TODO COMPUTE MEDIAN FILTER
    #
    for i in range(first, end):
        for j in range(ny):
            # Get 3x3 neighborhood around pixel p[i, j] with boundary checks
            neighbors = []
            for di in [-1, 0, 1]:
                for dj in [-1, 0, 1]:
                    ni = i + di
                    nj = j + dj
                    if 0 <= ni < nx and 0 <= nj < ny:  # Check boundaries
                        neighbors.append(buf[ni, nj])
            
            # Replace pixel value with the median of its neighbors
            new_buf[i - first, j] = np.median(neighbors)
    
    
    ##########################################
    #
    # RETURN LOCAL IMAGE PART
    #
    return part_id,new_buf

def main():
    data_dir = '/gext/jean-marc.gratien/BigDataHadoopSpark/TPs/data'
    file = os.path.join('lena_noisy.jpg')
    img_buf=readImg(file)
    print('SHAPE',img_buf.shape)
#     print('IMG\n',img_buf)
    nx=img_buf.shape[0]
    ny=img_buf.shape[1]
    
    ###########################################################################
    #
    # SEPARATE IMAGES INTO R, G, B CHANNELS
    r_img_buf = img_buf[:, :, 0]
    g_img_buf = img_buf[:, :, 1]
    b_img_buf = img_buf[:, :, 2]
    
    ###########################################################################
    #
    # SPLT IMAGES IN NB_PARTITIONS PARTS
    nb_partitions = 8
    print("NB PARTITIONS : ",nb_partitions)
    r_data = split_channel_images(r_img_buf, nx, nb_partitions)
    g_data = split_channel_images(g_img_buf, nx, nb_partitions)
    b_data = split_channel_images(b_img_buf, nx, nb_partitions)
    
    
    
    ###########################################################################
    #
    # CREATE SPARKCONTEXT
    sc =SparkContext()
    r_data_rdd = sc.parallelize(r_data,nb_partitions)
    g_data_rdd = sc.parallelize(g_data,nb_partitions)
    b_data_rdd = sc.parallelize(b_data,nb_partitions)

    
    
    ###########################################################################
    #
    # PARALLEL MEDIAN FILTER COMPUTATION
    r_result_rdd = r_data_rdd.map(part_median_filter)
    g_result_rdd = g_data_rdd.map(part_median_filter)
    b_result_rdd = b_data_rdd.map(part_median_filter)
    r_result_data = r_result_rdd.collect()
    g_result_data = g_result_rdd.collect()
    b_result_data = b_result_rdd.collect()

    ###########################################################################
    #
    # COMPUTE NEW IMAGE RESULTS FROM RESULT RDD
    # TODO
    r_new_img_buf = compute_image_from_rdd(r_result_data, nb_partitions, nx, ny)
    g_new_img_buf = compute_image_from_rdd(g_result_data, nb_partitions, nx, ny)
    b_new_img_buf = compute_image_from_rdd(b_result_data, nb_partitions, nx, ny)
    
    # Stop SparkContext
    sc.stop()
    
    # Reassemble filtered channels into a single image
    new_img_buf = np.stack((r_new_img_buf, g_new_img_buf, b_new_img_buf), axis=2)
#     print('NEW IMG\n',new_img_buf)
    print('CREATE NEW PICTURE FILE')
    filter_file = os.path.join('spark_lena_filter.jpg')
    writeImg(filter_file,new_img_buf)

if __name__ == '__main__':
    main()

/tmp/ipykernel_50945/2247019270.py:7: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(path)
/usr/local/lib/python3.9/dist-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found


SHAPE (128, 128, 3)
NB PARTITIONS :  8


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/10 19:12:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/10 19:12:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/03/10 19:12:39 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/03/10 19:12:39 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
new_buf = new_buf =  (16, 128)                                      (0 + 8) / 8]
 new_buf = (16, 128)new_buf =   
(16, 128)(16, 128)

new_buf =  (16, 128)
new_buf =  (16, 128)
new_buf = new_buf =   (16, 128)(16, 128)

new_buf =  (16, 128)                                                            
new_buf =  (16, 128)
new_buf =  (16, 128)
new_buf =  (16, 128)
new_buf = new_buf =  (16, 128)
 (16, 128)
new_buf =  (16, 128)
new_b

CREATE NEW PICTURE FILE


### D/ Dask : parallelisation of the image processing algorithm « MedianFilter »

In [2]:
import dask
from dask import delayed, compute
import imageio
import os
import numpy as np

def readImg(path):
    img = imageio.imread(path)
    im = np.array(img, dtype='uint8')
    return im

def writeImg(path, buf):
    imageio.imwrite(path, buf)

def split_channel_images(channel_img, nx, nb_partitions):
    data = []
    begin = 0
    block_size = nx // nb_partitions
    for ip in range(nb_partitions):
        end = min(begin + block_size, nx)
        data.append([ip, begin, end, channel_img])
        begin = end
    return data

def compute_image_from_parts(result_data, nb_partitions, nx, ny):
    new_img_buf = np.zeros((nx, ny), dtype='uint8')
    for part_id, part_buf in result_data:
        start = part_id * (nx // nb_partitions)
        end = start + part_buf.shape[0]
        new_img_buf[start:end, :] = part_buf
    return new_img_buf

@delayed
def part_median_filter(local_data):
    part_id = int(local_data[0])
    first = int(local_data[1])
    end = int(local_data[2])
    buf = local_data[3]
    nx = buf.shape[0]
    ny = buf.shape[1]

    new_buf = np.zeros((end - first, ny), dtype='uint8')

    for i in range(first, end):
        for j in range(ny):
            # Get 3x3 neighborhood around pixel p[i, j] with boundary checks
            neighbors = []
            for di in [-1, 0, 1]:
                for dj in [-1, 0, 1]:
                    ni = i + di
                    nj = j + dj
                    if 0 <= ni < nx and 0 <= nj < ny:  # Check boundaries
                        neighbors.append(buf[ni, nj])
            
            # Replace pixel value with the median of its neighbors
            new_buf[i - first, j] = np.median(neighbors)

    return part_id, new_buf

def main():
    data_dir = '/gext/jean-marc.gratien/BigDataHadoopSpark/TPs/data'
    file = os.path.join('lena_noisy.jpg')
    img_buf = readImg(file)
    print('SHAPE', img_buf.shape)

    nx = img_buf.shape[0]
    ny = img_buf.shape[1]

    # Separate images into R, G, B channels
    r_img_buf = img_buf[:, :, 0]
    g_img_buf = img_buf[:, :, 1]
    b_img_buf = img_buf[:, :, 2]

    # Split images into parts
    nb_partitions = 8
    print("NB PARTITIONS : ", nb_partitions)
    r_data = split_channel_images(r_img_buf, nx, nb_partitions)
    g_data = split_channel_images(g_img_buf, nx, nb_partitions)
    b_data = split_channel_images(b_img_buf, nx, nb_partitions)

    # Parallel median filter computation using Dask
    r_result = [part_median_filter(part) for part in r_data]
    g_result = [part_median_filter(part) for part in g_data]
    b_result = [part_median_filter(part) for part in b_data]

    # Execute the Dask tasks
    r_result_data, g_result_data, b_result_data = compute(r_result, g_result, b_result)

    # Reconstruct images from parts
    r_new_img_buf = compute_image_from_parts(r_result_data, nb_partitions, nx, ny)
    g_new_img_buf = compute_image_from_parts(g_result_data, nb_partitions, nx, ny)
    b_new_img_buf = compute_image_from_parts(b_result_data, nb_partitions, nx, ny)

    # Reassemble filtered channels into a single image
    new_img_buf = np.stack((r_new_img_buf, g_new_img_buf, b_new_img_buf), axis=2)
#     print('NEW IMG\n', new_img_buf)
    print('CREATE NEW PICTURE FILE')
    filter_file = os.path.join('dask_lena_filter.jpg')
    writeImg(filter_file, new_img_buf)

if __name__ == '__main__':
    main()

/tmp/ipykernel_50945/2223614160.py:8: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(path)


SHAPE (128, 128, 3)
NB PARTITIONS :  8
CREATE NEW PICTURE FILE
